# 04 — Tucker-2 Decomposition  ⚡ FAST — this is what you run live

Phase 1 (sensitivity sweep) is the only part with any real cost, and it's
scoped down here (few validation batches, 5 candidate ratios per layer) to
finish in a couple of minutes on CPU or GPU — designed to be run live in
front of the audience. Phase 2 (budget distribution + compression) itself
is nearly instant.

If you want Phase 1 even faster for the demo, drop `MAX_BATCHES` further or
pre-compute `sensitivities`/`sweep_data` once and just re-run Phase 2 live
with different `GLOBAL_TARGET` values to show the trade-off interactively.

In [ ]:
import sys, os, json, time
sys.path.append(os.path.abspath("../src"))
import torch
from torch.utils.data import DataLoader

from model import YOLOv3, count_conv_params
from loss import YOLOLoss
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_decompose import (get_compressible_conv_layers, phase1_sensitivity_sweep,
                               phase2_compress, distribute_budget)

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## Load the trained model

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)
model.load_state_dict(state["model_state"])
model.to(DEVICE)

total_before, conv_before = count_conv_params(model)
print(f"loaded trained model: {total_before:,} total params ({conv_before:,} conv params)")

## A small validation loader for the sensitivity sweep (doesn't need to be large)

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=2)
criterions = [YOLOLoss(NUM_CLASSES) for _ in range(3)]

layers = get_compressible_conv_layers(model)
print(f"{len(layers)} compressible conv layers (stem + prediction heads excluded)")

## Phase 1 — per-layer sensitivity

Sweeps 5 compression ratios per layer, fits a degree-4 (biquadratic)
polynomial to (ratio, val-loss-delta), and takes `sensitivity = 2*|a2|`
— the curvature of that fit. Low sensitivity = the layer's loss barely
reacts to compression = safe to compress hard.

In [ ]:
MAX_BATCHES = 4   # keep this small for a live demo; raise for a more accurate offline run
RATIOS = (0.9, 0.7, 0.5, 0.3, 0.1)

t0 = time.time()
sensitivities, sweep_data, baseline_loss = phase1_sensitivity_sweep(
    model, layers, val_loader, criterions, DEVICE, ratios=RATIOS, max_batches=MAX_BATCHES)
print(f"\nPhase 1 done in {time.time()-t0:.1f}s   baseline val loss: {baseline_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt
names = list(sensitivities.keys())
vals = [sensitivities[n] for n in names]
order = sorted(range(len(names)), key=lambda i: -vals[i])[:20]

plt.figure(figsize=(9, 6))
plt.barh([names[i] for i in order][::-1], [vals[i] for i in order][::-1])
plt.xlabel("sensitivity = 2*|a2|")
plt.title("Top-20 most sensitive layers (leave these closer to full rank)")
plt.tight_layout()
plt.show()

## Phase 2 — inverse-sensitivity budget distribution + sequential compression

`GLOBAL_TARGET` is the overall fraction of compressible parameters you want
to keep (0.5 = compress compressible conv layers to ~50% of their original
size, before/after guardrails). This is the number to change live during
the demo to show different compression/accuracy trade-offs.

In [ ]:
GLOBAL_TARGET = 0.5

per_layer_ratios = distribute_budget(sensitivities, GLOBAL_TARGET)
print("Per-layer target ratios (lower = compressed harder):")
for n, r in sorted(per_layer_ratios.items(), key=lambda kv: kv[1])[:10]:
    print(f"  {n:45s}  {r:.2f}")
print("  ...")

In [ ]:
compressed_model, compression_log = phase2_compress(
    model, layers, sensitivities, sweep_data, GLOBAL_TARGET,
    ckpt_dir="../checkpoints/decomposition", device=DEVICE)

## Save the compressed model

In [ ]:
torch.save(compressed_model.state_dict(), "../checkpoints/yolov3_compressed.pt")
with open("../checkpoints/compression_log.json", "w") as f:
    json.dump(compression_log, f, indent=2)

total_after, conv_after = count_conv_params(compressed_model)
print(f"conv params: {conv_before:,} -> {conv_after:,}  ({conv_after/conv_before:.1%} kept)")
print(f"total params: {total_before:,} -> {total_after:,}  ({total_after/total_before:.1%} kept)")
print("saved ../checkpoints/yolov3_compressed.pt")

Forward-pass sanity check on the compressed model — shapes should be identical to the original.

In [ ]:
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    outs = compressed_model(x)
print([tuple(o.shape) for o in outs])